In [1]:
!pip install ultralytics opencv-python


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import cv2
import os
import numpy as np
from ultralytics import YOLO
from pathlib import Path

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\bomma\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [12]:
# Setup paths
input_frames_folder = "saved_frames"  # From video_to_frames.ipynb
output_frames_folder = "detected_frames"

# Create output folder
os.makedirs(output_frames_folder, exist_ok=True)

print(f"Input frames folder: {input_frames_folder}")
print(f"Output frames folder: {output_frames_folder}")

# Check if input folder exists
if not os.path.exists(input_frames_folder):
    print(f"Error: {input_frames_folder} not found!")
else:
    frames_count = len(os.listdir(input_frames_folder))
    print(f"Found {frames_count} frames to process")

Input frames folder: saved_frames
Output frames folder: detected_frames
Found 85 frames to process


In [11]:
# Load YOLO11n model
print("Loading YOLO11n model...")
model = YOLO("yolo11n.pt")
print("✓ YOLO11n model loaded successfully")

# Person class ID in YOLO
PERSON_CLASS_ID = 0

Loading YOLO11n model...
✓ YOLO11n model loaded successfully


In [13]:
# Process frames with YOLO detection
frames_list = sorted(os.listdir(input_frames_folder))
total_frames = len(frames_list)
total_persons_detected = 0

print(f"\nProcessing {total_frames} frames...\n")

for idx, frame_file in enumerate(frames_list, 1):
    frame_path = os.path.join(input_frames_folder, frame_file)
    
    # Read frame
    frame = cv2.imread(frame_path)
    
    if frame is None:
        print(f"Error reading {frame_file}")
        continue
    
    # Run YOLO detection with lower confidence threshold to catch partial detections
    results = model(frame, conf=0.2)
    
    persons_in_frame = 0
    
    # Draw bounding boxes for detected persons
    for result in results:
        boxes = result.boxes
        
        for box in boxes:
            cls = int(box.cls[0])
            
            # Only draw boxes for person class (0)
            if cls == PERSON_CLASS_ID:
                confidence = float(box.conf[0])
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                
                # Draw green rectangle for bounding box (even for partial detections)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                
                # Draw label with confidence
                label = f"Person {confidence:.2f}"
                label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0]
                
                # Draw background for label
                cv2.rectangle(frame, (x1, y1 - label_size[1] - 5), 
                            (x1 + label_size[0], y1), (0, 255, 0), -1)
                
                # Draw text (probability)
                cv2.putText(frame, label, (x1, y1 - 5), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
                
                persons_in_frame += 1
                total_persons_detected += 1
    
    # Save frame with bounding boxes
    output_path = os.path.join(output_frames_folder, frame_file)
    cv2.imwrite(output_path, frame)
    
    if persons_in_frame > 0:
        print(f"[{idx}/{total_frames}] {frame_file}: {persons_in_frame} person(s) detected")
    else:
        print(f"[{idx}/{total_frames}] {frame_file}: No persons detected")

print(f"\n{'='*60}")
print(f"DETECTION COMPLETED")
print(f"{'='*60}")
print(f"Total frames processed: {total_frames}")
print(f"Total persons detected: {total_persons_detected}")
print(f"Output folder: {output_frames_folder}")
print(f"Confidence threshold: 0.2 (detects partial persons)")
print(f"{'='*60}")


Processing 85 frames...


0: 640x384 1 person, 1 fire hydrant, 87.4ms
Speed: 3.3ms preprocess, 87.4ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 384)
[1/85] frame_001_00-00-00.jpg: 1 person(s) detected

0: 640x384 2 persons, 80.8ms
Speed: 2.8ms preprocess, 80.8ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 384)
[2/85] frame_002_00-00-00.jpg: 2 person(s) detected

0: 640x384 1 person, 84.2ms
Speed: 2.6ms preprocess, 84.2ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 384)
[3/85] frame_003_00-00-01.jpg: 1 person(s) detected

0: 640x384 1 person, 1 donut, 87.7ms
Speed: 2.7ms preprocess, 87.7ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 384)
[4/85] frame_004_00-00-01.jpg: 1 person(s) detected

0: 640x384 1 person, 109.2ms
Speed: 2.8ms preprocess, 109.2ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 384)
[5/85] frame_005_00-00-02.jpg: 1 person(s) detected

0: 640x384 (no detections), 108.1ms
Speed: 3.0ms prepro

In [14]:
# Display sample results (first 5 frames)
detected_frames_list = sorted(os.listdir(output_frames_folder))

print(f"\nDisplaying first 5 detected frames:\n")

for i, frame_file in enumerate(detected_frames_list[:5], 1):
    frame_path = os.path.join(output_frames_folder, frame_file)
    
    img = cv2.imread(frame_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    print(f"{i}. {frame_file}")
    
    # Display frame
    cv2.imshow(f"Detection: {frame_file}", img)
    cv2.waitKey(1000)  # Show for 1 second

cv2.destroyAllWindows()
print("\nDone displaying frames!")


Displaying first 5 detected frames:

1. frame_001_00-00-00.jpg
2. frame_002_00-00-00.jpg
3. frame_003_00-00-01.jpg
4. frame_004_00-00-01.jpg
5. frame_005_00-00-02.jpg

Done displaying frames!
